# 02 — LFM2.5-VL: Vision-Finetuning mit QLoRA

Feintunt ein **Vision-Language-Modell** (Default `LiquidAI/LFM2.5-VL-1.6B`) auf
eigene Bild-Text-Paare.

**Datenformat** — flache Zeilen statt Konversationen, weil Bilder nicht ins JSONL
passen. `image` ist ein Pfad relativ zu `data.image_root`:

```json
{"image": "screenshots/panel-01.png", "question": "Welcher Wert steht bei Temperatur?", "answer": "63 °C"}
```

**Layout in Drive:**

```
MyDrive/muscal-lfm/vl/data/vl_train.jsonl
MyDrive/muscal-lfm/vl/data/images/...      # <- data.image_root
```

In [ ]:
# Colab brings torch; we only need the training stack. --upgrade on purpose:
# LFM2.5 checkpoints need a recent transformers.
!pip install -q -U transformers trl peft accelerate bitsandbytes datasets pyyaml

import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
    print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/muscal-lfm/vl')
(DRIVE_DIR / 'data').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'outputs').mkdir(parents=True, exist_ok=True)
print("drive project dir:", DRIVE_DIR)
print("contents:", sorted(p.name for p in DRIVE_DIR.iterdir()))

In [ ]:
# Get the training code. If you already have the repo in Drive, skip this cell.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/INDIEaner84/MUSCAL-ColabAPI-ProviderLLM.git"
REF = os.environ.get("MUSCAL_REF", "main")   # set to your branch before pushing

if not Path('/content/MUSCAL-ColabAPI-ProviderLLM').exists():
    !git clone --depth 1 --branch $REF $REPO_URL /content/MUSCAL-ColabAPI-ProviderLLM

%cd /content/MUSCAL-ColabAPI-ProviderLLM
sys.path.insert(0, '/content/MUSCAL-ColabAPI-ProviderLLM')
print("cwd:", Path.cwd())

## Bilder nach Colab holen

Am einfachsten als gezippter Ordner in Drive, dann hier entpacken. Bei sehr
vielen Bildern statt dessen direkt aus Drive lesen — Entpacken kostet Zeit, das
Zippen spart sie.

In [ ]:
import zipfile
from pathlib import Path

ZIP = DRIVE_DIR / "data" / "images.zip"
IMAGE_ROOT = DRIVE_DIR / "data" / "images"

if ZIP.exists() and not IMAGE_ROOT.exists():
    IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(IMAGE_ROOT)
    print("extracted ->", IMAGE_ROOT)

print("images found:", len(list(IMAGE_ROOT.rglob("*.[pj][np][g]"))) if IMAGE_ROOT.exists() else 0)

In [ ]:
from muscal_lfm.config import TrainConfig
from muscal_lfm.model import gpu_info
import yaml

CONFIG = "configs/vl_qlora.yaml"
cfg = TrainConfig.from_yaml(CONFIG)

# --- your settings -------------------------------------------------------
cfg.data.train_file = str(DRIVE_DIR / "data" / "train.jsonl")
cfg.training.output_dir = str(DRIVE_DIR / "outputs" / "vl.yaml")
cfg.data.train_file = str(DRIVE_DIR / "data" / "vl_train.jsonl")
cfg.data.image_root = str(IMAGE_ROOT)
cfg.data.max_image_tokens = 256     # bei OOM auf 128 / 64 senken
cfg.training.per_device_train_batch_size = 1
cfg.training.gradient_accumulation_steps = 16
# ------------------------------------------------------------------------

print("gpu:", gpu_info())
print()
print(cfg.to_yaml())

## Daten prüfen

Der VLM-Pfad baut die Chat-Template-Eingaben selbst; geprüft wird, dass pro Zeile
`image`, `question` und `answer` vorhanden sind und die Bilder ladbar sind.

In [ ]:
from datasets import Dataset
from muscal_lfm import data as data_utils
import json
from pathlib import Path

rows = [json.loads(l) for l in Path(cfg.data.train_file).read_text(encoding="utf-8").splitlines() if l.strip()]
ds = Dataset.from_list(rows)
print("rows:", len(ds), "| columns:", ds.column_names)
print("example:", rows[0])

problems = data_utils.validate(ds, flat_vl=True)
print("[ok] shape valid" if not problems else problems[:10])

# teurer Check: alle Bilder wirklich laden
convs = data_utils.build_vlm_conversations(ds.select(range(min(16, len(ds)))), image_root=cfg.data.image_root)
print("built", len(convs), "conversations (probe)")

In [ ]:
from muscal_lfm.train import train
from pathlib import Path

Path(cfg.training.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.training.output_dir, "config.resolved.yaml").write_text(cfg.to_yaml())

adapter_dir = train(cfg)
print("adapter:", adapter_dir)
print(sorted(p.name for p in Path(adapter_dir).iterdir()))

In [ ]:
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

processor = AutoProcessor.from_pretrained(cfg.model.id, trust_remote_code=True)
base = AutoModelForImageTextToText.from_pretrained(
    cfg.model.id, dtype="auto", device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base, str(adapter_dir))

# point this at one of your own images
image_path = sorted(Path(cfg.data.image_root).rglob("*.png"))[0] if cfg.data.image_root else None
if image_path:
    conv = [{
        "role": "user",
        "content": [
            {"type": "image", "image": Image.open(image_path).convert("RGB")},
            {"type": "text", "text": "Beschreibe, was auf diesem Bild zu sehen ist."},
        ],
    }]
    inputs = processor.apply_chat_template(conv, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    print(processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
from muscal_lfm.export import merge_adapter, export_gguf

merged = merge_adapter(
    cfg.model.id,
    adapter_dir,
    str(Path(cfg.training.output_dir) / "merged"),
    track=cfg.model.track,
)
print("merged:", merged)

# Optional: GGUF for llama.cpp / Ollama / LM Studio.
# Needs a current llama.cpp (LFM2.5 is a young architecture) and ~10 min.
EXPORT_GGUF = False
if EXPORT_GGUF:
    gguf = export_gguf(merged, quant="q4_k_m")
    !cp "$gguf" "$DRIVE_DIR/outputs/"

## Hinweise

* **Bild-Token sind der Speicherhebel.** `max_image_tokens` 256 → 64 spart mehr als
  jede Batch-Size-Änderung.
* Die kleinen VLMs sind laut Liquid für **enge Anwendungsfälle** gedacht — ein
  feingetuntes 450M schlägt ein generisches 3B auf *seiner* Aufgabe oft.
* Bilder müssen RGB sein; `build_vlm_conversations` konvertiert, aber CMYK- oder
  16-bit-TIFFs vorher selbst sauber machen.